In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
# %%capture --no-stderr
# !pip install python-dotenv langchain-community langchain-core langchain langchain-openai langchain-chroma

In [3]:
# 환경변수 설정

In [4]:
# 라이브러리 불러오기
from dotenv import load_dotenv
import os
from langchain_openai import OpenAI

In [5]:
# .env 파일에서 환경 변수 로드 (.env 파일에는 OPENAI API 키값을 적으면 됩니다. -> OPENAI_API_KEY=...)
load_dotenv("/content/.env")
# 환경 변수에서 API 키 가져오기
api_key = os.getenv("OPENAI_API_KEY")
# 오픈AI 대규모 언어 모델 초기화
llm = OpenAI()

In [6]:
# <이전 대화를 포함한 메시지 전달>

# 라이브러리 불러오기
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

chat = ChatOpenAI(model="gpt-4o-mini")

# 프롬프트 템플릿 정의: 금융 상담 역할
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 금융 상담사입니다. 사용자에게 최선의 금융 조언을 제공합니다."),
        ("placeholder", "{messages}"),  # 대화 이력 추가
    ]
)

# 프롬프트와 모델을 연결하여 체인 생성
chain = prompt | chat

# 이전 대화를 포함한 메시지 전달
ai_msg = chain.invoke(
    {
        "messages": [
            ("human", "저축을 늘리기 위해 무엇을 할 수 있나요?"),  # 사용자의 첫 질문
            ("ai", "저축 목표를 설정하고, 매달 자동 이체로 일정 금액을 저축하세요."),  # 챗봇의 답변
            ("human", "방금 뭐라고 했나요?"),  # 사용자의 재확인 질문
        ],
    }
)
print(ai_msg.content)  # 챗봇의 응답 출력


저축을 늘리기 위해 저축 목표를 설정하고, 매달 자동 이체로 일정 금액을 저축하는 것이 좋다고 말씀드렸습니다. 이렇게 하면 저축 습관을 기르기 쉽고, 필요한 경우 재정 계획을 더 잘 관리할 수 있습니다. 추가로, 불필요한 지출을 줄이고, 예산을 세우는 것도 도움이 됩니다.


In [7]:
# <`ChatMessageHistory`를 사용한 메시지 관리>

from langchain_community.chat_message_histories import ChatMessageHistory

# 대화 이력 저장을 위한 클래스 초기화
chat_history = ChatMessageHistory()

# 사용자 메시지 추가
chat_history.add_user_message("저축을 늘리기 위해 무엇을 할 수 있나요?")
chat_history.add_ai_message("저축 목표를 설정하고, 매달 자동 이체로 일정 금액을 저축하세요.")

# 새로운 질문 추가 후 다시 체인 실행
chat_history.add_user_message("방금 뭐라고 했나요?")
ai_response = chain.invoke({"messages": chat_history.messages})
print(ai_response.content)  # 챗봇은 이전 메시지를 기억하여 답변합니다.

저축을 늘리기 위해 목표를 설정하고, 매달 자동 이체를 통해 일정 금액을 저축하는 것을 추천했습니다. 이렇게 하면 매달 저축하는 것이 더 수월해집니다. 추가적으로, 지출을 검토하고 불필요한 지출을 줄이는 것도 좋은 방법입니다.


In [8]:
# <` RunnableWithMessageHistory`를 사용한 메시지 관리>

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# 시스템 메시지와 대화 이력을 사용하는 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 금융 상담사입니다. 모든 질문에 최선을 다해 답변하십시오."),
        ("placeholder", "{chat_history}"),  # 이전 대화 이력
        ("human", "{input}"),  # 사용자의 새로운 질문
    ]
)

# 대화 이력을 관리할 체인 설정
chat_history = ChatMessageHistory()
chain = prompt | chat

# RunnableWithMessageHistory 클래스를 사용해 체인을 감쌉니다
chain_with_message_history = RunnableWithMessageHistory(
    chain,
    lambda session_id: chat_history,  # 세션 ID에 따라 대화 이력을 불러오는 함수
    input_messages_key="input",  # 입력 메시지의 키 설정
    history_messages_key="chat_history",  # 대화 이력의 키 설정
)

# RunnableWithMessageHistory 클래스를 사용해 체인을 감쌉니다
chain_with_message_history = RunnableWithMessageHistory(
    chain,
    lambda session_id: chat_history,  # 세션 ID에 따라 대화 이력을 불러오는 함수
    input_messages_key="input",  # 입력 메시지의 키 설정
    history_messages_key="chat_history",  # 대화 이력의 키 설정
)

# 질문 메시지 체인 실행
chain_with_message_history.invoke(
    {"input": "저축을 늘리기 위해 무엇을 할 수 있나요?"},
    {"configurable": {"session_id": "unused"}},
).content


d:\WorkSpace\Python\langchain-tutorial\Ch01. Langchain Basics\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)
d:\WorkSpace\Python\langchain-tutorial\Ch01. Langchain Basics\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


'저축을 늘리기 위해 다음과 같은 방법들을 고려해볼 수 있습니다:\n\n1. **예산 수립**: 매월 수입과 지출을 상세히 기록하여 예산을 세우세요. 이를 통해 불필요한 지출을 줄이고 저축할 수 있는 금액을 확인할 수 있습니다.\n\n2. **자동 이체 설정**: 급여날에 자동으로 저축 계좌로 특정 금액이 이체되도록 설정하세요. 이렇게 하면 저축을 잊지 않고 자연스럽게 늘릴 수 있습니다.\n\n3. **소비 습관 점검**: 필요한 소비와 불필요한 소비를 구분해보고, 불필요한 지출 항목을 줄이는 주의를 기울이세요. 예를 들어, 외식, 커피, 취미활동 비용 등을 점검해 보세요.\n\n4. **할인 및 쿠폰 활용**: 할인이나 쿠폰을 적극적으로 활용하여 소비를 줄이고, 절약한 비용을 저축하세요.\n\n5. **부가 수입 창출**: 본업 외에 아르바이트, 프리랜서 일, 판매 등을 통해 추가 수입을 창출해 그 금액을 저축하세요.\n\n6. **금융 상품 활용**: 이자율이 더 높은 저축 계좌나 적금, 투자 상품 등을 찾아보세요. 적절한 금융 상품을 통해 저축에 대한 수익을 극대화할 수 있습니다.\n\n7. **목표 설정**: 구체적인 저축 목표(예: 여행, 집 구매, 긴급 자금 등)를 정하고, 이를 달성하기 위한 계획을 세우면 동기부여가 됩니다.\n\n8. **지출 추적 앱 사용**: 지출 관리 앱을 사용하여 소비 패턴을 분석하고 필요한 부분을 개선해보세요.\n\n저축을 늘리는 것은 시간이 걸리는 과정이지만, 꾸준한 노력과 계획을 통해 원하는 목표를 이룰 수 있습니다.'

In [9]:
# 새로운 입력 메시지를 추가하고 체인을 실행
chain_with_message_history.invoke(
    {"input": "내가 방금 뭐라고 했나요?"},  # 사용자의 질문
    {"configurable": {"session_id": "unused"}}  # 세션 ID 설정
).content


'당신은 저축을 늘리기 위해 무엇을 할 수 있는지에 대해 질문하셨습니다. 저는 저축을 늘리기 위한 여러 가지 방법을 제안했습니다. 도움이 더 필요하시거나 다른 질문이 있으시면 말씀해 주세요!'

In [10]:
# <메시지 트리밍 예제>

# 라이브러리 불러오기
from langchain_core.messages import trim_messages
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter

# 메시지 트리밍 유틸리티 설정
trimmer = trim_messages(strategy="last", max_tokens=2, token_counter=len)

# 트리밍된 대화 이력과 함께 체인 실행
chain_with_trimming = (
    RunnablePassthrough.assign(chat_history=itemgetter("chat_history") | trimmer)
    | prompt
    | chat
)

# 트리밍된 대화 이력을 사용하는 체인 설정
chain_with_trimmed_history = RunnableWithMessageHistory(
    chain_with_trimming,
    lambda session_id: chat_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

# 새로운 대화 내용 추가 후 체인 실행
chain_with_trimmed_history.invoke(
    {"input": "저는 5년 내에 집을 사기 위해 어떤 재정 계획을 세워야 하나요?"},  # 사용자의 질문
    {"configurable": {"session_id": "finance_session_1"}}  # 세션 ID 설정
)



AIMessage(content='5년 내에 집을 사기 위한 재정 계획을 세우는 것은 중요한 단계입니다. 아래에 제안하는 몇 가지 단계를 참고해 보세요:\n\n1. **목표 설정**:\n   - 구매할 집의 가격대와 위치를 결정하세요. 예를 들어, 5년 후 구매하고자 하는 집의 평균가격을 조사해서 대략적인 금액을 설정합니다.\n\n2. **예산 수립**:\n   - 현재의 소득과 지출을 분석해 보세요. 월별 예산을 작성하여 얼마나 저축할 수 있는지 파악합니다.\n   - 불필요한 지출을 줄이고 저축 비율을 높이는 방법을 고려합니다.\n\n3. **저축 계획**:\n   - 매달 저축할 금액을 정하세요. 목표 금액에 도달하기 위해 필요한 월 저축 금액을 계산합니다.\n   - 저축 계좌를 개설하고 정기적으로 기여하세요. 고이율 저축계좌나 금융상품을 고려해볼 수 있습니다.\n\n4. **신용 점수 관리**:\n   - 주택 담보 대출을 받기 위해 신용 점수를 확인하고 관리하세요. 신용 점수를 높여 대출 조건을 유리하게 만드는 것이 중요합니다.\n\n5. **부채 관리**:\n   - 기존의 부채를 관리하고 가능한 한 빨리 상환하세요. 높은 이자율의 부채는 우선적으로 처리하는 것이 좋습니다.\n\n6. **투자 고려**:\n   - 저축 외에 남는 자금을 주식이나 Fonds와 같은 투자에 활용해 더 높은 수익을 추구할 수 있습니다. 그러나 투자에는 리스크가 따르므로 신중하게 접근해야 합니다.\n\n7. **모기지 대출 준비**:\n   - 다양한 대출 상품을 조사하여 금리와 조건을 비교하세요. 대출 사전 승인 신청도 미리 해보는 것이 좋습니다.\n   - 다운페이먼트(계약금)를 얼마나 준비해야 하는지 알아보고 그에 맞춰 저축 계획을 세우세요.\n\n8. **전문가 상담**:\n   - 필요에 따라 금융 상담사나 부동산 전문가의 도움을 받을 수 있습니다. 그들은 상황에 맞춘 조언을 제공할 수 있습니다.\n\n이 계획을 바탕으로 체계적으로 움직인다면 5년 내에 집을 사

In [11]:
# 새로운 입력 메시지를 추가하고 체인을 실행
chain_with_trimmed_history.invoke(
    {"input": "내가 방금 뭐라고 했나요?"},  # 사용자의 질문
    {"configurable": {"session_id": "finance_session_1"}}  # 세션 ID 설정
).content


'당신은 "저는 5년 내에 집을 사기 위해 어떤 재정 계획을 세워야 하나요?"라고 질문하셨습니다. 이 질문에 대해서 제가 재정 계획을 세우는 방법에 대한 여러 단계를 제안해 드렸습니다. 다시 질문이 있으시면 알려주세요!'

In [12]:
# <이전 대화 요약 내용 기반으로 답변하기>

def summarize_messages(chain_input):
    stored_messages = chat_history.messages
    if len(stored_messages) == 0:
        return False
    # 대화를 요약하기 위한 프롬프트 템플릿 설정
    summarization_prompt = ChatPromptTemplate.from_messages(
        [
            ("placeholder", "{chat_history}"),  # 이전 대화 이력
            (
                "user",
                "이전 대화를 요약해 주세요. 가능한 한 많은 세부 정보를 포함하십시오.",  # 요약 요청 메시지
            ),
        ]
    )

    # 요약 체인 생성 및 실행
    summarization_chain = summarization_prompt | chat
    summary_message = summarization_chain.invoke({"chat_history": stored_messages})
    chat_history.clear()  # 요약 후 이전 대화 삭제
    chat_history.add_message(summary_message)  # 요약된 메시지를 대화 이력에 추가
    return True

In [13]:
# 대화 요약을 처리하는 체인 설정
chain_with_summarization = (
    RunnablePassthrough.assign(messages_summarized=summarize_messages)
    | chain_with_message_history
)

# 요약된 대화를 기반으로 새로운 질문에 응답
print(chain_with_summarization.invoke(
    {"input": "저에게 어떤 재정적 조언을 해주셨나요?"},  # 사용자의 질문
    {"configurable": {"session_id": "unused"}}  # 세션 ID 설정
).content)


재정적 조언으로는 다음과 같은 내용을 말씀드렸습니다:

1. **저축 전략**:
   - **예산 수립**: 자신의 수입과 지출을 분석하여 예산을 세워 불필요한 지출을 줄이고 저축을 늘림.
   - **자동 이체 설정**: 급여일에 저축 계좌로 자동 이체를 설정하여 저축을 습관화함.
   - **소비 습관 점검**: 필요한 소비와 불필요한 소비를 구분하여 절약.
   - **할인 및 쿠폰 활용**: 소비 시 할인이나 쿠폰을 적극 활용하여 비용 절감.
   - **부가 수입 창출**: 아르바이트나 프리랜서 작업 등의 추가 소득을 생성.

2. **집 구매 계획**:
   - **목표 설정**: 구매할 집의 가격대와 location을 정하여 대략적인 목표 금액 결정.
   - **예산 수립**: 현재 소득과 지출을 분석해 매달 저축할 금액 파악.
   - **저축 계획**: 목표 금액을 기준으로 매월 저축할 금액 계산.
   - **신용 점수 관리**: 주택 담보 대출을 받을 때 유리한 신용 점수를 유지.
   - **부채 관리**: 기존 부채를 줄여 대출에 대한 부담을 최소화.
   - **투자 고려**: 저축 외에 여유 자금을 투자하여 수익을 극대화.
   - **모기지 대출 준비**: 다양한 대출 상품을 조사하고 대출에 대한 사전 승인을 받음.
   - **전문가 상담**: 필요시 금융 상담사나 부동산 전문가의 조언을 받는 것.

이러한 조언들이 여러분의 재정 목표 달성에 유용하길 바랍니다. 추가 질문이나 보다 구체적인 조언이 필요하시면 언제든지 말씀해 주세요!
